**Nome:** Valdir de Souza Carvalho Neto

**Turma:** SI 241

In [2]:
!pip install gurobipy

import gurobipy as gp
from gurobipy import GRB
from math import sqrt, pow
print(f"Gurobi: {gp.gurobi.version()}")

Gurobi: (13, 0, 3)


### MODELAGEM MATEMÁTICA

**Função Objetivo (Minimizar a distância total percorrida):**
$$
\min Z = \sum_{i \in V} \sum_{\substack{j \in V \\ j \neq i}} c_{ij} x_{ij
}$$

**Sujeito a:**

*Garantia de visita única (Exatamente 1 veículo entra em cada cliente):*
$$
\sum_{\substack{i \in V \\ i \neq j}} x_{ij} = 1 \quad \forall j \in C
$$

*Garantia de saída única (Exatamente 1 veículo sai de cada cliente):*
$$
\sum_{\substack{j \in V \\ j \neq i}} x_{ij} = 1 \quad \forall i \in C
$$

*Saída e retorno dos veículos ao depósito (Exatamente 2 veículos são utilizados):*
$$
\begin{aligned}
\sum_{j \in C} x_{0j} &= 2 \\ \sum_{i \in C} x_{i0} &= 2
\end{aligned}
$$

*Eliminação de sub-rotas e restrição de capacidade (Adaptação MTZ):*
$$
u_i - u_j + Q x_{ij} \leq Q - d_j \quad \forall i, j \in C, i \neq j
$$

*Carga mínima acumulada ao visitar um cliente:*
$$
u_i \geq d_i \quad \forall i \in C
$$

*Domínio das Variáveis:*
$$
\begin{aligned} x_{ij} &\in \{0, 1\} \quad \forall i, j \in V \\ u_i &\in [0, Q] \quad \forall i \in C \end{aligned}
$$

In [4]:
dist_x = [0, 1, 2, 4, 5, 6, 7, 8, 9]
dist_y = [0, 2, 5, 4, 1, 5, 2, 4, 1]

n_pontos = len(dist_x)
V = range(n_pontos)
clientes = list(range(1, n_pontos))

demandas = [0, 1, 2, 3, 2, 1, 4, 1, 2]
Q = 8

dist_x_y = [[0] * n_pontos for _ in V]
for i in V:
  for j in V:
    dist_x_y[i][j] = sqrt(pow(dist_x[i] - dist_x[j], 2) + pow(dist_y[i] - dist_y[j], 2))

m = gp.Model("roteamento")
m.Params.OutputFlag = 0
x = m.addVars(V, V, vtype=GRB.BINARY, name="x")
u = m.addVars(clientes, lb=0, ub=Q, vtype=GRB.CONTINUOUS, name="carga_acumulada")
m.setObjective(gp.quicksum(dist_x_y[i][j] * x[i, j] for i in V for j in V if i != j), GRB.MINIMIZE)

for i in clientes:
  for j in clientes:
    if i != j:
      m.addConstr(u[i] - u[j] + Q * x[i, j] <= Q - demandas[j])

for i in clientes:
    m.addConstr(u[i] >= demandas[i])

for i in clientes:
    m.addConstr(gp.quicksum(x[i, j] for j in V if i != j) == 1)

for j in clientes:
    m.addConstr(gp.quicksum(x[i, j] for i in V if i != j) == 1)

m.addConstr(gp.quicksum(x[0, j] for j in clientes) == 2)
m.addConstr(gp.quicksum(x[i, 0] for i in clientes) == 2)

m.optimize()

if m.Status != GRB.OPTIMAL:
  print(f"O modelo não encontrou solução ótima. Status: {m.Status}")
else:
  print(f"\nDistância mínima total: {m.ObjVal:.2f}")

  arcos_ativos = [(i, j) for i in V for j in V if i != j and x[i, j].X > 0.5]
  saidas_deposito = [j for (i, j) in arcos_ativos if i == 0]

  print("\nRotas detalhadas:")
  for idx, proximo_no in enumerate(saidas_deposito, start=1):
    rota = [0, proximo_no]
    atual = proximo_no
    while atual != 0:
      for (origem, destino) in arcos_ativos:
        if origem == atual:
         rota.append(destino)
         atual = destino
         break

    caminho_formatado = " -> ".join(map(str, rota))
    print(f"Veículo {idx}: {caminho_formatado}")



Distância mínima total: 38.62

Rotas detalhadas:
Veículo 1: 0 -> 4 -> 3 -> 2 -> 1 -> 0
Veículo 2: 0 -> 6 -> 8 -> 7 -> 5 -> 0
